In [8]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

In [9]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# 1. Generate Dummy Data for Driving Mode

# Physics conversions for PSI
pedal_contact_area_sq_in = 6.0 # Example: 6 square inches of foot contact
force_kg = np.random.uniform(0, 40, 1000) # 0 to 40 kg of force applied
force_lbf = force_kg * 2.20462 # Convert mass (kg) to force (lbs)
pressure_psi = force_lbf / pedal_contact_area_sq_in # Calculate PSI

driving_data = {
    'fsr': np.random.randint(0, 100, 1000),
    'accel_x': np.random.rand(1000) * 9.8,
    'accel_y': np.random.rand(1000) * 9.8,
    'accel_z': np.random.rand(1000) * 9.8,
    'gyro_x': np.random.rand(1000) * 200,
    'gyro_y': np.random.rand(1000) * 200,
    'gyro_z': np.random.rand(1000) * 200,
    'pattern': np.random.choice(['smooth', 'aggressive', 'stop_and_go'], 1000),
    'pedal_state': np.random.choice(['left_brake', 'right_accelerate', 'neutral'], 1000),
    'force_kg': force_kg,
    'pressure_psi': pressure_psi
}
df_driving = pd.DataFrame(driving_data)

# 2. Preprocess the Data
pattern_encoder = LabelEncoder()
pedal_encoder = LabelEncoder()

df_driving['pattern_encoded'] = pattern_encoder.fit_transform(df_driving['pattern'])
df_driving['pedal_encoded'] = pedal_encoder.fit_transform(df_driving['pedal_state'])

features_driving = ['fsr', 'accel_x', 'accel_y', 'accel_z', 'gyro_x', 'gyro_y', 'gyro_z']
scaler_driving = MinMaxScaler()
df_driving[features_driving] = scaler_driving.fit_transform(df_driving[features_driving])

display(df_driving.head())
print(f"Pattern Classes: {pattern_encoder.classes_}")
print(f"Pedal Classes: {pedal_encoder.classes_}")

,fsr,accel_x,accel_y,accel_z,gyro_x,gyro_y,gyro_z,pattern,pedal_state,force_kg,pressure_psi,pattern_encoded,pedal_encoded
0,0.777778,0.905673,0.153393,0.188615,0.148458,0.697385,0.610574,aggressive,neutral,26.424601,9.709367,0,1
1,0.767677,0.842318,0.825900,0.999611,0.761554,0.604947,0.503293,aggressive,neutral,12.917448,4.746344,0,1
2,1.000000,0.859544,0.644203,0.501190,0.965608,0.127864,0.221963,aggressive,left_brake,7.665746,2.816676,0,0
3,0.616162,0.708252,0.749759,0.412565,0.046411,0.572854,0.142915,stop_and_go,left_brake,18.088265,6.646292,2,0
4,0.060606,0.570861,0.923549,0.431073,0.776315,0.232519,0.045680,stop_and_go,neutral,33.385872,12.267194,2,1


Pattern Classes: ['aggressive' 'smooth' 'stop_and_go']
Pedal Classes: ['left_brake' 'neutral' 'right_accelerate']


In [11]:

# 3. Create Sequences for Triple-Output Model
def create_driving_sequences(data_df, feature_cols, window_size, step_size=1):
    X_seq, y_pat, y_ped, y_pres = [], [], [], []
    data_values = data_df[feature_cols].values

    for i in range(0, len(data_values) - window_size, step_size):
        X_seq.append(data_values[i:i + window_size])
        y_pat.append(data_df['pattern_encoded'].values[i + window_size - 1])
        y_ped.append(data_df['pedal_encoded'].values[i + window_size - 1])
        # Target is now the specific PSI value
        y_pres.append(data_df['pressure_psi'].values[i + window_size - 1])

    return np.array(X_seq), np.array(y_pat), np.array(y_ped), np.array(y_pres)

window_size_driving = 5
X_drv, y_pat, y_ped, y_psi = create_driving_sequences(df_driving, features_driving, window_size_driving)

# One-hot encode categoricals
y_pat_ohe = to_categorical(y_pat, num_classes=len(pattern_encoder.classes_))
y_ped_ohe = to_categorical(y_ped, num_classes=len(pedal_encoder.classes_))

# Train/Test Split
X_train_d, X_test_d, y_pat_train, y_pat_test, y_ped_train, y_ped_test, y_psi_train, y_psi_test = train_test_split(
    X_drv, y_pat_ohe, y_ped_ohe, y_psi, test_size=0.2, random_state=42
)

print(f"Driving X_train shape: {X_train_d.shape}")

Driving X_train shape: (796, 5, 7)


In [12]:
from tensorflow.keras.layers import Input, Dense, Dropout, Flatten
from tensorflow.keras.models import Model

# 4. Build Triple-Output Time-Series DNN Architecture
inputs_drv = Input(shape=(window_size_driving, len(features_driving)), name='driving_sensor_inputs')

# Shared Representation: Flatten the sequence, then pass through Dense layers (DNN)
x_drv = Flatten()(inputs_drv)
x_drv = Dense(units=128, activation='relu')(x_drv)
x_drv = Dropout(0.2)(x_drv)
x_drv = Dense(units=64, activation='relu')(x_drv)
x_drv = Dropout(0.2)(x_drv)

# Output 1: Driving Pattern (Classification)
pattern_out = Dense(len(pattern_encoder.classes_), activation='softmax', name='pattern_output')(x_drv)

# Output 2: Left/Right Pedal State (Classification)
pedal_out = Dense(len(pedal_encoder.classes_), activation='softmax', name='pedal_output')(x_drv)

# Output 3: Pressure Limit (Regression in PSI)
psi_out = Dense(1, activation='linear', name='psi_output')(x_drv)

# Define Model
driving_model = Model(inputs=inputs_drv, outputs=[pattern_out, pedal_out, psi_out])

# Compile Model
driving_model.compile(
    optimizer='adam',
    loss={
        'pattern_output': 'categorical_crossentropy',
        'pedal_output': 'categorical_crossentropy',
        'psi_output': 'mse'  # Mean Squared Error for PSI
    },
    loss_weights={
        'pattern_output': 1.0,
        'pedal_output': 1.0,
        'psi_output': 0.05 # Scaled weight for PSI regression
    },
    metrics={
        'pattern_output': 'accuracy',
        'pedal_output': 'accuracy',
        'psi_output': 'mae'
    }
)

driving_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ driving_sensor_inp… │ (None, 5, 7)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 35)        │          0 │ driving_sensor_i… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │      4,608 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pattern_output      │ (None, 3)         │        195 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pedal_output        │ (None, 3)         │        195 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ psi_output (Dense)  │ (None, 1)         │         65 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 13,319 (52.03 KB)

 Trainable params: 13,319 (52.03 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# 5. Train the Driving Model with PSI targets
history_driving = driving_model.fit(
    X_train_d,
    {
        'pattern_output': y_pat_train,
        'pedal_output': y_ped_train,
        'psi_output': y_psi_train
    },
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Evaluate the model
eval_drv = driving_model.evaluate(
    X_test_d,
    {'pattern_output': y_pat_test, 'pedal_output': y_ped_test, 'psi_output': y_psi_test},
    verbose=0
)
print(f"\nEvaluation Results:\n{eval_drv}")


Epoch 1/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - loss: 5.1531 - pattern_output_accuracy: 0.3145 - pattern_output_loss: 1.1503 - pedal_output_accuracy: 0.3318 - pedal_output_loss: 1.2444 - psi_output_loss: 55.0739 - psi_output_mae: 6.1208 - val_loss: 4.0289 - val_pattern_output_accuracy: 0.3625 - val_pattern_output_loss: 1.1282 - val_pedal_output_accuracy: 0.3125 - val_pedal_output_loss: 1.2055 - val_psi_output_loss: 33.9050 - val_psi_output_mae: 4.8728
Epoch 2/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 3.7060 - pattern_output_accuracy: 0.3569 - pattern_output_loss: 1.1867 - pedal_output_accuracy: 0.3066 - pedal_output_loss: 1.2530 - psi_output_loss: 25.2795 - psi_output_mae: 4.1829 - val_loss: 3.1131 - val_pattern_output_accuracy: 0.3562 - val_pattern_output_loss: 1.0914 - val_pedal_output_accuracy: 0.3375 - val_pedal_output_loss: 1.1006 - val_psi_output_loss: 18.4204 - val_psi_output_mae: 3.7330
Epoch 3/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 3.4685 - pattern_outp